In [1]:
# ============================================================
# COMPLETE MCP DEMO IN ONE GOOGLE COLAB / JUPYTER CELL
# ============================================================

import sys
import subprocess
import textwrap
from pathlib import Path


In [2]:
# ------------------------------------------------------------
# 1. INSTALL MCP
# ------------------------------------------------------------
print("Installing MCP Python SDK...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "mcp[cli]>=1.27,<2"
    ],
    check=True
)

print("MCP installation completed.\n")

Installing MCP Python SDK...
MCP installation completed.



In [4]:
# ------------------------------------------------------------
# 2. CREATE MCP SERVER PROGRAM
# ------------------------------------------------------------
server_code = r'''
from mcp.server.fastmcp import FastMCP


# Create the MCP server.
mcp = FastMCP("Vehicle Telemetry MCP Server")


# ------------------------------------------------------------
# MCP TOOL 1: Engine-temperature diagnosis
# ------------------------------------------------------------
@mcp.tool()
def check_engine_temperature(temperature_c: float) -> dict:
    """
    Analyse engine coolant temperature.

    Args:
        temperature_c:
            Current engine coolant temperature in Celsius.
    """

    if temperature_c >= 115:
        return {
            "status": "critical",
            "temperature_c": temperature_c,
            "message": (
                "Stop the vehicle safely and inspect the coolant level, "
                "radiator fan, thermostat and water pump."
            )
        }

    if temperature_c >= 105:
        return {
            "status": "warning",
            "temperature_c": temperature_c,
            "message": (
                "Engine temperature is high. Check coolant level, "
                "radiator fan and thermostat."
            )
        }

    return {
        "status": "normal",
        "temperature_c": temperature_c,
        "message": "Engine temperature is within the normal range."
    }


# ------------------------------------------------------------
# MCP TOOL 2: Brake-pressure diagnosis
# ------------------------------------------------------------
@mcp.tool()
def check_brake_pressure(
    pressure_bar: float,
    vehicle_speed_kmph: float
) -> dict:
    """
    Analyse brake pressure and determine the safety risk.

    Args:
        pressure_bar:
            Current hydraulic brake pressure in bar.

        vehicle_speed_kmph:
            Current vehicle speed in kilometres per hour.
    """

    if pressure_bar < 40 and vehicle_speed_kmph > 0:
        return {
            "status": "critical",
            "pressure_bar": pressure_bar,
            "vehicle_speed_kmph": vehicle_speed_kmph,
            "action": (
                "Stop the vehicle safely and immobilize it "
                "for brake-system inspection."
            )
        }

    if pressure_bar < 55:
        return {
            "status": "warning",
            "pressure_bar": pressure_bar,
            "vehicle_speed_kmph": vehicle_speed_kmph,
            "action": "Inspect the hydraulic brake system."
        }

    return {
        "status": "normal",
        "pressure_bar": pressure_bar,
        "vehicle_speed_kmph": vehicle_speed_kmph,
        "action": "No immediate brake-system action is required."
    }


# ------------------------------------------------------------
# MCP TOOL 3: Average calculation
# ------------------------------------------------------------
@mcp.tool()
def calculate_average(
    first_value: float,
    second_value: float
) -> float:
    """
    Calculate the average of two telemetry readings.
    """

    return (first_value + second_value) / 2


# ------------------------------------------------------------
# MCP RESOURCE 1: Cooling-system manual
# ------------------------------------------------------------
@mcp.resource("vehicle://manual/cooling")
def cooling_manual() -> str:
    """
    Return cooling-system maintenance information.
    """

    return """
VEHICLE COOLING-SYSTEM MANUAL

Normal range:
Engine coolant temperature is normally approximately 85°C to 104°C.

Warning condition:
At 105°C or above, inspect the coolant level, radiator fan,
thermostat, coolant pump, radiator and coolant hoses.

Critical condition:
At 115°C or above, stop the vehicle safely and arrange
an immediate cooling-system inspection.
"""


# ------------------------------------------------------------
# MCP RESOURCE 2: Brake-system manual
# ------------------------------------------------------------
@mcp.resource("vehicle://manual/brakes")
def brake_manual() -> str:
    """
    Return brake-system maintenance information.
    """

    return """
VEHICLE BRAKE-SYSTEM MANUAL

Normal brake pressure:
Brake pressure should remain above the safe operating threshold.

Warning condition:
Below 55 bar, inspect the hydraulic brake system.

Critical condition:
Below 40 bar while the vehicle is moving, stop the vehicle safely
and immobilize it until the brake system is inspected.
"""


# ------------------------------------------------------------
# DYNAMIC MCP RESOURCE
# ------------------------------------------------------------
@mcp.resource("vehicle://profile/{vehicle_id}")
def vehicle_profile(vehicle_id: str) -> str:
    """
    Return a demonstration vehicle profile.
    """

    return f"""
Vehicle ID: {vehicle_id}
Vehicle type: Fleet demonstration vehicle
Service status: Active
Last service: 30 days ago
"""


# ------------------------------------------------------------
# MCP PROMPT
# ------------------------------------------------------------
@mcp.prompt()
def diagnostic_prompt(
    fault: str,
    severity: str = "unknown"
) -> str:
    """
    Create a reusable automotive diagnostic prompt.
    """

    return f"""
Act as an automotive diagnostic engineer.

Fault:
{fault}

Severity:
{severity}

Explain:

1. Probable causes
2. Immediate safety action
3. Components that should be inspected
4. Whether human escalation is required
"""


# Run using STDIO transport.
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

server_path = Path("/tmp/vehicle_mcp_server.py")
server_path.write_text(
    textwrap.dedent(server_code),
    encoding="utf-8"
)

print("Created MCP server:", server_path)


# ------------------------------------------------------------
# 3. CREATE MCP CLIENT PROGRAM
# ------------------------------------------------------------
client_code = r'''
import asyncio
import json
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def print_separator(title: str) -> None:
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def print_tool_result(result) -> None:
    """
    Print text and structured output returned by an MCP tool.
    """

    if getattr(result, "structuredContent", None):
        print(
            json.dumps(
                result.structuredContent,
                indent=2,
                ensure_ascii=False
            )
        )
        return

    for item in result.content:
        if hasattr(item, "text"):
            print(item.text)
        else:
            print(item)


async def main() -> None:

    # Tell the MCP client how to start the MCP server.
    server_parameters = StdioServerParameters(
        command=sys.executable,
        args=["/tmp/vehicle_mcp_server.py"]
    )

    # Start the MCP server as a subprocess.
    async with stdio_client(server_parameters) as streams:

        read_stream, write_stream = streams

        # Establish an MCP session.
        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            # MCP initialization handshake.
            await session.initialize()

            print_separator("1. MCP CONNECTION ESTABLISHED")
            print("The MCP client successfully connected to the server.")


            # ------------------------------------------------
            # DISCOVER TOOLS
            # ------------------------------------------------
            print_separator("2. DISCOVER AVAILABLE MCP TOOLS")

            tools_result = await session.list_tools()

            for number, tool in enumerate(
                tools_result.tools,
                start=1
            ):
                print(f"\nTool {number}")
                print("Name       :", tool.name)
                print("Description:", tool.description)
                print(
                    "Input schema:",
                    json.dumps(
                        tool.inputSchema,
                        indent=2
                    )
                )


            # ------------------------------------------------
            # CALL ENGINE TOOL
            # ------------------------------------------------
            print_separator("3. CALL ENGINE-TEMPERATURE TOOL")

            engine_result = await session.call_tool(
                "check_engine_temperature",
                {
                    "temperature_c": 118
                }
            )

            print_tool_result(engine_result)


            # ------------------------------------------------
            # CALL BRAKE TOOL
            # ------------------------------------------------
            print_separator("4. CALL BRAKE-PRESSURE TOOL")

            brake_result = await session.call_tool(
                "check_brake_pressure",
                {
                    "pressure_bar": 36,
                    "vehicle_speed_kmph": 62
                }
            )

            print_tool_result(brake_result)


            # ------------------------------------------------
            # CALL CALCULATOR TOOL
            # ------------------------------------------------
            print_separator("5. CALL AVERAGE-CALCULATION TOOL")

            average_result = await session.call_tool(
                "calculate_average",
                {
                    "first_value": 102,
                    "second_value": 110
                }
            )

            print_tool_result(average_result)


            # ------------------------------------------------
            # DISCOVER RESOURCES
            # ------------------------------------------------
            print_separator("6. DISCOVER AVAILABLE MCP RESOURCES")

            resources_result = await session.list_resources()

            for number, resource in enumerate(
                resources_result.resources,
                start=1
            ):
                print(f"\nResource {number}")
                print("URI :", resource.uri)
                print("Name:", resource.name)


            # ------------------------------------------------
            # READ COOLING RESOURCE
            # ------------------------------------------------
            print_separator("7. READ COOLING-MANUAL RESOURCE")

            cooling_result = await session.read_resource(
                "vehicle://manual/cooling"
            )

            for content in cooling_result.contents:
                if hasattr(content, "text"):
                    print(content.text)
                else:
                    print(content)


            # ------------------------------------------------
            # READ BRAKE RESOURCE
            # ------------------------------------------------
            print_separator("8. READ BRAKE-MANUAL RESOURCE")

            brake_manual_result = await session.read_resource(
                "vehicle://manual/brakes"
            )

            for content in brake_manual_result.contents:
                if hasattr(content, "text"):
                    print(content.text)
                else:
                    print(content)


            # ------------------------------------------------
            # READ DYNAMIC RESOURCE
            # ------------------------------------------------
            print_separator("9. READ DYNAMIC VEHICLE PROFILE")

            vehicle_result = await session.read_resource(
                "vehicle://profile/KA01-FLEET-009"
            )

            for content in vehicle_result.contents:
                if hasattr(content, "text"):
                    print(content.text)
                else:
                    print(content)


            # ------------------------------------------------
            # DISCOVER PROMPTS
            # ------------------------------------------------
            print_separator("10. DISCOVER AVAILABLE MCP PROMPTS")

            prompts_result = await session.list_prompts()

            for number, prompt in enumerate(
                prompts_result.prompts,
                start=1
            ):
                print(f"\nPrompt {number}")
                print("Name       :", prompt.name)
                print("Description:", prompt.description)


            # ------------------------------------------------
            # GET PROMPT
            # ------------------------------------------------
            print_separator("11. GET REUSABLE DIAGNOSTIC PROMPT")

            prompt_result = await session.get_prompt(
                "diagnostic_prompt",
                arguments={
                    "fault": "Engine coolant temperature is 118 degrees Celsius",
                    "severity": "critical"
                }
            )

            for message in prompt_result.messages:
                content = message.content

                if hasattr(content, "text"):
                    print(content.text)
                else:
                    print(content)


            print_separator("12. MCP DEMONSTRATION COMPLETED")

            print("""
This demonstration showed:

1. An MCP client connecting to an MCP server.
2. Automatic discovery of MCP tools.
3. Calling remote tools through MCP.
4. Automatic discovery of MCP resources.
5. Reading static and dynamic resources.
6. Automatic discovery of reusable MCP prompts.
7. Retrieving a reusable prompt from the server.
""")


if __name__ == "__main__":
    asyncio.run(main())
'''

client_path = Path("/tmp/vehicle_mcp_client.py")
client_path.write_text(
    textwrap.dedent(client_code),
    encoding="utf-8"
)

print("Created MCP client:", client_path)


# ------------------------------------------------------------
# 4. RUN CLIENT AS A REAL SUBPROCESS
# ------------------------------------------------------------
print("\nStarting the MCP demonstration...\n")

completed_process = subprocess.run(
    [
        sys.executable,
        str(client_path)
    ],
    capture_output=True,
    text=True,
    timeout=120
)


# ------------------------------------------------------------
# 5. DISPLAY OUTPUT
# ------------------------------------------------------------
if completed_process.stdout:
    print(completed_process.stdout)

if completed_process.stderr:
    print("\nSERVER / CLIENT DIAGNOSTIC OUTPUT:")
    print(completed_process.stderr)

if completed_process.returncode != 0:
    raise RuntimeError(
        "The MCP demonstration failed. "
        f"Return code: {completed_process.returncode}"
    )

print("Program finished successfully.")

Created MCP server: /tmp/vehicle_mcp_server.py
Created MCP client: /tmp/vehicle_mcp_client.py

Starting the MCP demonstration...


1. MCP CONNECTION ESTABLISHED
The MCP client successfully connected to the server.

2. DISCOVER AVAILABLE MCP TOOLS

Tool 1
Name       : check_engine_temperature
Description: 
    Analyse engine coolant temperature.

    Args:
        temperature_c:
            Current engine coolant temperature in Celsius.
    
Input schema: {
  "properties": {
    "temperature_c": {
      "title": "Temperature C",
      "type": "number"
    }
  },
  "required": [
    "temperature_c"
  ],
  "title": "check_engine_temperatureArguments",
  "type": "object"
}

Tool 2
Name       : check_brake_pressure
Description: 
    Analyse brake pressure and determine the safety risk.

    Args:
        pressure_bar:
            Current hydraulic brake pressure in bar.

        vehicle_speed_kmph:
            Current vehicle speed in kilometres per hour.
    
Input schema: {
  "properties"